In [ ]:
import os
import sys
import json
ROOT_DIR = '/root/Desktop/data/private/DIG-dig-stable'
sys.path.append(ROOT_DIR)
sys.path.insert(0, ROOT_DIR)
from models.model_graph_mil import *
from models.H2GCNmodel import *

from datasets.BatchWSI import BatchWSI
import torch
import torch.nn as nn
from torch_geometric.data import Data
import dig.xgraph.method.pgexplainer as pgexp_mod
from torch_geometric.nn.conv.message_passing import MessagePassing
pgexp_mod.MessagePassing = MessagePassing

from dig.xgraph.method import PGExplainer
import copy

In [ ]:
def load_graphs_by_patient_ids(graph_dir, patient_ids, suffix="-01Z-00-DX1.pt", map_func=None):
    data_list = []
    missing = []

    for pid in patient_ids:
        fname = map_func(pid) if map_func is not None else f"{pid}{suffix}"
        fpath = os.path.join(graph_dir, fname)
        if not os.path.exists(fpath):
            missing.append(fpath)
            continue

        g = torch.load(fpath, map_location="cpu")

        assert hasattr(g, "x"), f"{pid} graph missing x"
        assert hasattr(g, "edge_index"), f"{pid} graph missing edge_index"
        assert hasattr(g, "edge_latent"), f"{pid} graph missing edge_latent"

        data_list.append(g)

    if len(missing) > 0:
        print("[WARN] Missing graph files:")
        for m in missing:
            print("  ", m)

    return data_list


class BranchModelForPGExplainer(torch.nn.Module):

    def __init__(self, base_model, device, branch_name=None, edge_type="spatial"):
        super().__init__()
        self.m = base_model
        self.device = device
        self.branch_name = branch_name
        self.edge_type = edge_type

    def _sync_pyg_masks(self):
        for mp in self.modules():
            if isinstance(mp, MessagePassing):
                em = getattr(mp, "__edge_mask__", None)
                if em is not None:
                    mp._edge_mask = em
                    mp._apply_sigmoid = True
                    if hasattr(mp, "_explain"):
                        mp._explain = True

    def _select_branch(self):
        if self.branch_name is None:
            return self.m
        else:
            return getattr(self.m, self.branch_name)


    def _select_edge(self, data=None, edge_index=None, edge_latent=None):
        if data is not None:
            if self.edge_type == "spatial":
                return data.edge_index
            elif self.edge_type == "latent":
                return data.edge_latent
            else:
                raise ValueError(f"Unknown edge_type: {self.edge_type}")
        else:
            if self.edge_type == "spatial":
                return edge_index
            elif self.edge_type == "latent":
                return edge_latent if edge_latent is not None else edge_index
            else:
                raise ValueError(f"Unknown edge_type: {self.edge_type}")

    def forward(self, x=None, edge_index=None, edge_latent=None, data=None, **kwargs):
        if data is not None:
            x = data.x

        use_edge = self._select_edge(data=data, edge_index=edge_index, edge_latent=edge_latent)

        assert x is not None, "x is required"
        assert use_edge is not None, f"{self.edge_type} edge is required"

        x = x.to(self.device)
        use_edge = use_edge.to(self.device)

        d = Data(x=x, edge_index=use_edge)
        x_path = BatchWSI.from_data_list([d]).to(self.device)

        self._sync_pyg_masks()

        branch_model = self._select_branch()
        out = branch_model(x_path=x_path)

        logits = out["logits"] if isinstance(out, dict) else out[3]
        return logits

    def get_emb(self, x, edge_index=None, edge_latent=None, data=None):
        if data is not None:
            x = data.x
        return x.to(self.device).detach()


@torch.no_grad()
def infer_pg_in_channels(wrapped_model, sample_data, device):
    wrapped_model.eval()

    x = sample_data.x.to(device)
    ei = sample_data.edge_index.to(device) if hasattr(sample_data, "edge_index") else None
    el = sample_data.edge_latent.to(device) if hasattr(sample_data, "edge_latent") else None

    emb = wrapped_model.get_emb(x=x, edge_index=ei, edge_latent=el)
    emb_dim = emb.size(1)
    in_channels = 2 * emb_dim
    return in_channels, emb_dim


def train_pgexplainer_multi_patient(
    wrapped_model,
    train_graphs,
    test_graphs,
    device,
    save_dir,
    epochs=30,
    lr=0.003,
    coff_size=0.01,
    coff_ent=5e-4,
    t0=5.0,
    t1=1.0
):
    os.makedirs(save_dir, exist_ok=True)

    wrapped_model = wrapped_model.to(device).eval()

    in_channels, emb_dim = infer_pg_in_channels(wrapped_model, train_graphs[0], device)
    print(f"[PGExplainer] emb_dim={emb_dim}, in_channels={in_channels}")

    pg = PGExplainer(
        model=wrapped_model,
        in_channels=in_channels,
        device=device,
        explain_graph=True,
        epochs=epochs,
        lr=lr,
        coff_size=coff_size,
        coff_ent=coff_ent,
        t0=t0,
        t1=t1
    ).to(device)

    pg.train()
    pg.train_explanation_network(train_graphs)

    ckpt = {
        "pgexplainer_state": pg.state_dict(),
        "in_channels": in_channels,
        "emb_dim": emb_dim,
        "epochs": epochs,
        "lr": lr,
        "coff_size": coff_size,
        "coff_ent": coff_ent,
        "t0": t0,
        "t1": t1,
        "num_train": len(train_graphs),
        "num_test": len(test_graphs),
    }
    torch.save(ckpt, os.path.join(save_dir, "pgexplainer_ckpt.pt"))
    print(f"[Saved] {os.path.join(save_dir, 'pgexplainer_ckpt.pt')}")

    return pg, wrapped_model


def get_adaptive_topk(num_edges, ratio=0.05, k_min=30, k_max=200):

    k = int(num_edges * ratio)
    k = max(k, k_min)
    k = min(k, k_max)
    k = min(k, num_edges)
    return k


@torch.no_grad()
def explain_on_dataset(
    pgexplainer,
    wrapped_model,
    dataset,
    device,
    patient_ids=None,
    topk=200,
    adaptive_topk=False,
    topk_ratio=0.05,
    topk_min=100,
    topk_max=800
):
    pgexplainer.eval()
    wrapped_model.eval()

    if patient_ids is not None:
        assert len(patient_ids) == len(dataset), \
            f"patient_ids length {len(patient_ids)} != dataset length {len(dataset)}"

    results = []

    for i, data in enumerate(dataset):
        patient_id = patient_ids[i] if patient_ids is not None else None

        x = data.x.to(device)
        ei = data.edge_index.to(device) if hasattr(data, "edge_index") else None
        el = data.edge_latent.to(device) if hasattr(data, "edge_latent") else None

        logits = wrapped_model(x=x, edge_index=ei, edge_latent=el)
        pred = int(logits.argmax(dim=-1))

        emb = wrapped_model.get_emb(x=x, edge_index=ei, edge_latent=el)

        explain_edge = ei if wrapped_model.edge_type == "spatial" else el

        ret = pgexplainer.explain(x, explain_edge, embed=emb, target=pred)

        if isinstance(ret, (tuple, list)) and len(ret) == 2:
            prob, edge_mask = ret
        else:
            prob, edge_mask = None, ret

        edge_mask = edge_mask.detach().cpu()
        explain_edge_cpu = explain_edge.detach().cpu()

        num_edges = edge_mask.numel()

        if adaptive_topk:
            k = get_adaptive_topk(
                num_edges=num_edges,
                ratio=topk_ratio,
                k_min=topk_min,
                k_max=topk_max,
            )
        else:
            k = min(topk, num_edges)

        top_idx = torch.topk(edge_mask, k=k, largest=True).indices
        top_ei = explain_edge_cpu[:, top_idx]

        results.append({
            "idx": i,
            "patient_id": patient_id,
            "pred": pred,
            "logits": logits.detach().cpu(),
            "edge_mask": edge_mask,
            "edge_index": explain_edge_cpu,
            "top_edge_index": top_ei,
            "topk_used": k,
            "num_edges": num_edges,
            "edge_type": wrapped_model.edge_type,
            "branch_name": wrapped_model.branch_name,
        })

        print(
            f"[sample {i}] patient_id={patient_id}, "
            f"edge_type={wrapped_model.edge_type}, topk_used={k}, pred={pred}"
        )

    return results

def make_explainer_wrapper(model, device, model_type="combined", branch_name=None, edge_type="spatial"):

    if model_type == "combined":
        assert branch_name is None or branch_name in ["patch_gcn_surv", "h2gcn"], \
            f"Invalid branch_name for combined model: {branch_name}"

        return BranchModelForPGExplainer(
            base_model=model,
            device=device,
            branch_name=branch_name,
            edge_type=edge_type
        )

    elif model_type == "single":
        return BranchModelForPGExplainer(
            base_model=model,
            device=device,
            branch_name=None,
            edge_type=edge_type
        )

    else:
        raise ValueError(f"Unknown model_type: {model_type}")


def remap_dataset_for_explainer(dataset, edge_type="spatial"):
    new_dataset = []
    for data in dataset:
        d = copy.copy(data)
        if edge_type == "spatial":
            d.edge_index = data.edge_index
        elif edge_type == "latent":
            assert hasattr(data, "edge_latent"), "data has no edge_latent"
            d.edge_index = data.edge_latent
        else:
            raise ValueError(f"Unknown edge_type: {edge_type}")
        new_dataset.append(d)
    return new_dataset

In [ ]:
import pandas as pd
splits_dir = '/root/Desktop/data/private/LIHC/5foldcv/lihc_343/splits_3.csv'
df = pd.read_csv(splits_dir)
train_ids = (df["train"].dropna().tolist() + df["validation"].dropna().tolist())
test_ids = df["test"].dropna().tolist()

In [ ]:
graph_dir = "/root/Desktop/data/private/LIHC/patch_graph_343_uni2/"
device = torch.device("cuda:2" if torch.cuda.is_available() else "cpu")

train_graphs = load_graphs_by_patient_ids(graph_dir, train_ids)
test_graphs  = load_graphs_by_patient_ids(graph_dir, test_ids)

# CombinedModel
# model_dict1 = {'num_layers': 4, 'edge_agg': ['spatial', 'spatial'], 'resample': 0, 'n_classes': 4}
# model_dict2 = {'feat_dim': 1536, 'hidden_dim': 128, 'class_dim': 4, 'edge_agg':['spatial', 'spatial']}
# model_1 = PatchGCN_Surv(**model_dict1)
# model_2 = H2GCN(**model_dict2)
# model = CombinedModel(model_1,model_2)
# args_dir = '/root/Desktop/data/private/hjx_product/results_final_0614/5foldcv/CombinedModel_nll_surv_a0.0_5foldcv_gc32_ss/lihc_CombinedModel_nll_surv_a0.0_5foldcv_gc32_s42/s_2_minloss_checkpoint.pt'

# patchgcn
# model_dict = {'num_layers': 4, 'edge_agg': ['spatial', None], 'resample': 0, 'n_classes': 4}
# model = PatchGCN_Surv(**model_dict)
# args_dir = '/root/Desktop/data/private/hjx_product/results_check/5foldcv/PatchGCN_nll_surv_a0.0_5foldcv_gc32_spatial/lihc_PatchGCN_nll_surv_a0.0_5foldcv_gc32_s42/s_2_minloss_checkpoint.pt'

# h2gcn
model_dict = {'feat_dim': 1536, 'hidden_dim': 128, 'class_dim': 4, 'edge_agg':[None, 'spatial']}
model = H2GCN(**model_dict)
args_dir = '/root/Desktop/data/private/hjx_product/results_check/5foldcv/H2GCN_nll_surv_a0.0_5foldcv_gc32_spatial/lihc_H2GCN_nll_surv_a0.0_5foldcv_gc32_s42/s_2_minloss_checkpoint.pt'

model.load_state_dict(torch.load(args_dir))
model = model.to(device).eval()

In [ ]:
save_dir = "/root/Desktop/data/private/hjx_product/results/H2GCN_spatial"

# model_type: "single", "combined"
# branch_name: "patch_gcn_surv", "h2gcn", None
wrapped_model = make_explainer_wrapper(
    model=model,
    device=device,
    model_type="single",
    branch_name=None,
    edge_type="spatial"
)

pg, wrapped = train_pgexplainer_multi_patient(
    wrapped_model=wrapped_model,
    train_graphs=train_graphs,
    test_graphs=test_graphs,
    device=device,
    save_dir=save_dir,
    epochs=30,
    lr=0.003
)

test_explanations = explain_on_dataset(pg, wrapped, test_graphs, device, patient_ids=test_ids, adaptive_topk=True, topk_ratio=0.05, topk_min=100, topk_max=800)
train_explanations = explain_on_dataset(pg, wrapped, train_graphs, device, patient_ids=train_ids,adaptive_topk=True, topk_ratio=0.05, topk_min=100, topk_max=800)

torch.save(test_explanations, os.path.join(save_dir, "test_explanations.pt"))
torch.save(train_explanations, os.path.join(save_dir, "train_explanations.pt"))

In [ ]:
import matplotlib.pyplot as plt

def plot_pg_edges_only(
    data,
    edge_mask,
    edge_topk,
    node_s_bg=6, node_s_hi=20,
    node_alpha_bg=0.12, node_alpha_hi=0.95,
    edge_lw=1.6,
    edge_alpha_min=0.25, edge_alpha_max=0.98,
    edge_color="black",
    title="PGExplainer top-K edges",
    invert_y=True,
    save_path=None,
    dpi=300,
):
    c = data.centroid.detach().cpu()
    ei = data.edge_index.detach().cpu()
    w = edge_mask.detach().cpu().float().view(-1)

    assert ei.size(1) == w.numel(), f"E mismatch: edge_index {ei.size(1)} vs edge_mask {w.numel()}"

    K = min(edge_topk, w.numel())
    top_idx = torch.topk(w, k=K, largest=True).indices

    nodes = set()
    for i in top_idx.tolist():
        u = int(ei[0, i]); v = int(ei[1, i])
        nodes.add(u); nodes.add(v)
    nodes = sorted(nodes)
    nodes_t = torch.tensor(nodes, dtype=torch.long)

    w_top = w[top_idx]
    w_min, w_max = float(w_top.min()), float(w_top.max())
    denom = (w_max - w_min) if (w_max > w_min) else 1.0
    alphas = (w_top - w_min) / denom
    alphas = edge_alpha_min + (edge_alpha_max - edge_alpha_min) * alphas
    alphas = alphas.clamp(0, 1).tolist()

    fig = plt.figure(figsize=(12, 12), facecolor="white")
    ax = plt.gca()
    ax.set_facecolor("white")

    plt.scatter(c[:, 0], c[:, 1], s=node_s_bg, alpha=node_alpha_bg, color="gray")
    plt.scatter(c[nodes_t, 0], c[nodes_t, 1], s=node_s_hi, alpha=node_alpha_hi, color="orange")

    for j, i in enumerate(top_idx.tolist()):
        u = int(ei[0, i]); v = int(ei[1, i])
        plt.plot([c[u, 0], c[v, 0]], [c[u, 1], c[v, 1]],
                 linewidth=edge_lw, alpha=alphas[j], color=edge_color)

    plt.title(title)
    plt.axis("equal")
    plt.axis("off")
    if invert_y:
        plt.gca().invert_yaxis()

    if save_path is not None:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=dpi, bbox_inches="tight", facecolor="white")

    plt.close(fig)

In [ ]:
import os
import torch

@torch.no_grad()
def visualize_test_set(
    pgexplainer,
    wrapped_model,
    ids,
    data_graphs,
    device,
    out_dir,
    topk_ratio=0.15,
    topk_min=200,
    topk_max=800,
    # topk_edges=200,
    prefix="test",
):
    os.makedirs(out_dir, exist_ok=True)
    pgexplainer.eval()
    wrapped_model.eval()

    for pid, data in zip(ids, data_graphs):
        x = data.x.to(device)
        ei = data.edge_index.to(device)
        logits = wrapped_model(x, ei)         # [1, C]
        pred = int(logits.argmax(dim=-1))

        emb = wrapped_model.get_emb(x, ei)    # [N, D]
        ret = pgexplainer.explain(x, ei, embed=emb, target=pred)
        if isinstance(ret, (tuple, list)) and len(ret) == 2:
            prob, edge_mask = ret
        else:
            edge_mask = ret

        save_pt = os.path.join(out_dir, f"{prefix}_{pid}_pred{pred}_edge_mask.pt")
        torch.save({
            "pred": pred,
            "logits": logits.detach().cpu(),
            "edge_mask": edge_mask.detach().cpu(),
            "edge_index": data.edge_index.detach().cpu()
        }, save_pt)

        num_edges = edge_mask.numel()
        K = get_adaptive_topk(num_edges=num_edges, ratio=topk_ratio, k_min=topk_min, k_max=topk_max)

        save_png = os.path.join(out_dir, f"{pid}_pred{pred}_top{K}.png")
        plot_pg_edges_only(
            data=data,
            edge_mask=edge_mask,
            edge_topk=K,
            edge_color="black",
            title=f"{prefix}_{pid}  pred={pred}  top{K} edges",
            save_path=save_png
        )

        print(f"[OK] saved: {save_png}")

In [ ]:
out_dir = "/root/Desktop/data/private/hjx_product/results/H2GCN_spatial/vis_test"
visualize_test_set(pg, wrapped, test_ids, test_graphs, device, out_dir, topk_ratio=0.05, topk_min=100, topk_max=800, prefix="patient")
visualize_test_set(pg, wrapped, train_ids, train_graphs, device, out_dir, topk_ratio=0.05, topk_min=100, topk_max=800, prefix="patient")

In [ ]:
import torch
import pandas as pd
import os

@torch.no_grad()
def predict_probs(wrapped_model, x, edge_index, device):
    logits = wrapped_model(x.to(device), edge_index.to(device))  # [1, C]
    probs = torch.softmax(logits, dim=-1)                        # [1, C]
    return probs, logits

@torch.no_grad()
def eval_metrics_pg_topk(
    wrapped_model,
    data,
    edge_mask,
    target_class,
    device,
    topk_edges=200
):
    x = data.x
    ei = data.edge_index
    N = x.size(0)
    E = ei.size(1)

    # --- select top-k edges ---
    w = edge_mask.detach().cpu().float().view(-1)
    k = min(topk_edges, w.numel())
    top_idx = torch.topk(w, k=k, largest=True).indices  # [k]

    ei_only = ei[:, top_idx]  # only explanation edges

    # removed = remove these top-k edges
    keep_mask = torch.ones(E, dtype=torch.bool)
    keep_mask[top_idx] = False
    ei_removed = ei[:, keep_mask]

    # --- probabilities ---
    p_full, _ = predict_probs(wrapped_model, x, ei, device)
    p_only, _ = predict_probs(wrapped_model, x, ei_only, device)
    p_removed, _ = predict_probs(wrapped_model, x, ei_removed, device)

    p_full_t = float(p_full[0, target_class].item())
    p_only_t = float(p_only[0, target_class].item())
    p_removed_t = float(p_removed[0, target_class].item())

    # removed = nodes of these top-k edges
    nodes_sub = torch.unique(ei[:, top_idx].reshape(-1))
    x_removed_node = x.clone()
    x_removed_node[nodes_sub] = 0
    p_removed_node, _ = predict_probs(wrapped_model, x_removed_node, ei, device)
    p_removed_node_t = float(p_removed_node[0, target_class].item())

    # --- node/edge sparsity (by selected top-k edges) ---
    nodes_sub = torch.unique(ei[:, top_idx].reshape(-1)).numel()
    V_s = int(nodes_sub)
    E_s = int(k)

    metrics = {
        "p_full": p_full_t,
        "p_only": p_only_t,
        "p_removed": p_removed_t,
        "fidelity_insertion": p_only_t,
        "fidelity_deletion": p_full_t - p_removed_t,
        "p_removed_node": p_removed_node_t,
        "fidelity_deletion_node": p_full_t - p_removed_node_t,
        "sparsity_nodes": 1.0 - (V_s / float(N)),
        "sparsity_edges": 1.0 - (E_s / float(E)),
        "V": int(N),
        "E": int(E),
        "V_s": int(V_s),
        "E_s": int(E_s),
    }
    return metrics

@torch.no_grad()
def evaluate_test_set_pg(
    pgexplainer,
    wrapped_model,
    ids,
    data_graphs,
    device,
    out_dir,
    topk_ratio=0.15,
    topk_min=100,
    topk_max=800,
    prefix="test"
):
    os.makedirs(out_dir, exist_ok=True)
    pgexplainer.eval()
    wrapped_model.eval()

    rows = []

    for i, data in enumerate(data_graphs):
        name = ids[i] if ids is not None else f"{prefix}_{i}"

        # ---- prediction ----
        probs_full, logits = predict_probs(wrapped_model, data.x, data.edge_index, device)
        pred = int(torch.argmax(probs_full, dim=-1).item())

        # ---- explain (need embed) ----
        emb = wrapped_model.get_emb(data.x.to(device), data.edge_index.to(device))
        ret = pgexplainer.explain(
            data.x.to(device),
            data.edge_index.to(device),
            embed=emb,
            target=pred
        )
        if isinstance(ret, (tuple, list)) and len(ret) == 2:
            prob, edge_mask = ret
        else:
            edge_mask = ret

        num_edges = edge_mask.numel()
        topk_edges = get_adaptive_topk(num_edges=num_edges, ratio=topk_ratio, k_min=topk_min, k_max=topk_max)

        # ---- metrics ----
        m = eval_metrics_pg_topk(
            wrapped_model=wrapped_model,
            data=data,
            edge_mask=edge_mask,
            target_class=pred,
            device=device,
            topk_edges=topk_edges
        )

        row = {"id": name, "pred": pred, "topk_edges": int(min(topk_edges, data.edge_index.size(1)))}
        row.update(m)
        rows.append(row)

        torch.save(
            {"id": name, "pred": pred, "logits": logits.cpu(),
             "edge_mask": edge_mask.detach().cpu(),
             "metrics": row},
            os.path.join(out_dir, f"{name}_pred{pred}_metrics.pt")
        )

    df = pd.DataFrame(rows)

    return df

In [ ]:
out_dir = "/root/Desktop/data/private/hjx_product/results/H2GCN_spatial/eval_test"
df_test = evaluate_test_set_pg(
    pgexplainer=pg,
    wrapped_model=wrapped,
    ids=test_ids,
    data_graphs=test_graphs,
    device=device,
    out_dir=out_dir,
    topk_ratio=0.05,
    topk_min=100,
    topk_max=800
)
csv_path_test = os.path.join(out_dir, f"pg_metrics_topk_test.csv")
df_test.to_csv(csv_path_test, index=False)

df_train = evaluate_test_set_pg(
    pgexplainer=pg,
    wrapped_model=wrapped,
    ids=train_ids,
    data_graphs=train_graphs,
    device=device,
    out_dir=out_dir,
    topk_ratio=0.05,
    topk_min=100,
    topk_max=800
)
csv_path_train = os.path.join(out_dir, f"pg_metrics_topk_train.csv")
df_train.to_csv(csv_path_train, index=False)